# P5 — Multi-Format Healthcare Clinical Manual Ingestion & Basic RAG
Educational retrieval demo only. The included clinical text is synthetic and must not be used for patient care.

In [38]:
!pip -q install sentence-transformers faiss-cpu pypdf

In [42]:
from pathlib import Path

doc_path = Path(
    "/kaggle/input/datasets/babblu123/"
    "real-world-ml-genai-kaggle-complete/"
    "real_world_ml_genai_lab/docs/p5_clinical_guidelines.md"
)

BASE_DIR = doc_path.parent.parent
DOCS_DIR = BASE_DIR / "docs"
DATA_DIR = BASE_DIR / "datasets"
EXAMPLES_DIR = BASE_DIR / "examples"

print("Project directory:", BASE_DIR)
print("Documents:", [p.name for p in DOCS_DIR.iterdir()])
print("Datasets:", [p.name for p in DATA_DIR.iterdir()])

Project directory: /kaggle/input/datasets/babblu123/real-world-ml-genai-kaggle-complete/real_world_ml_genai_lab
Documents: ['p10_logistics_policy.md', 'p5_clinical_guidelines.md', 'p5_sample_note.txt', 'p6_contracts.txt', 'p7_market_outlook.txt']
Datasets: ['p10_eval_scenarios.csv', 'p10_shipments.csv', 'p1_iot_telemetry.csv', 'p2_fintech_transactions.csv', 'p3_supply_demand.csv', 'p4_support_tickets.csv', 'p6_legal_queries.csv', 'p7_portfolios.csv', 'p9_claims.jsonl']


In [44]:
texts = []

for p in DOCS_DIR.iterdir():
    if p.is_file() and p.suffix.lower() in [".txt", ".md"]:
        content = p.read_text(encoding="utf-8", errors="ignore").strip()
        if content:
            texts.append((p.name, content))

def chunk_text(text, size=500, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = []

for source, content in texts:
    for i, chunk in enumerate(chunk_text(content)):
        chunks.append({
            "source": source,
            "chunk_id": i,
            "text": chunk
        })

print("Documents loaded:", len(texts))
print("Total chunks:", len(chunks))

if chunks:
    print("First chunk:", chunks[0]["text"])

Documents loaded: 5
Total chunks: 10
First chunk: # Synthetic Logistics Policy
## Delivery Service Levels
Standard shipments target delivery within 72 hours of dispatch. Express shipments target delivery within 24 hours, subject to route and weather conditions.
## Delay Escalation
A shipment delayed more than 12 hours should be flagged for operations review. Delays exceeding 24 hours require notifying the customer and documenting a recovery plan.
## Exceptions and Data Privacy
Exceptions require an owner, timestamp, reason code, and next action


In [45]:
from sentence_transformers import SentenceTransformer
import numpy as np
model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb=model.encode([c["text"] for c in chunks],normalize_embeddings=True,show_progress_bar=True)
def retrieve(query,k=3):
    q=model.encode([query],normalize_embeddings=True)[0]
    scores=emb @ q
    ids=np.argsort(scores)[::-1][:k]
    return [(float(scores[i]),chunks[i]) for i in ids]
for score,item in retrieve("What are red flags that require urgent assessment?"):
    print(round(score,3),item["source"],item["text"][:240].replace("\n"," "))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

0.232 p5_clinical_guidelines.md # Sample Clinical Knowledge Base (Educational Only)  ## Fever in Adults Adults with fever should monitor temperature, hydration, and associated symptoms. Seek urgent medical assessment for severe breathing difficulty, confusion, stiff neck,
0.227 p5_clinical_guidelines.md t, dose, allergies, kidney and liver considerations, and possible interactions before medication use. A qualified clinician or pharmacist should verify medication decisions. Do not infer a dose from this sample document.  ## Follow-up and E
0.225 p5_sample_note.txt Sample note: A clinician should review allergies and current medications before treatment. This synthetic note is for retrieval experiments only.


## Evaluation
Try at least five questions. Record whether the retrieved chunks contain the answer and compare fixed-size chunking with header-based chunking. This notebook demonstrates retrieval, not validated medical advice.